# Week 4 Query Parser Evaluation

This notebook reviews the Week 4 query parser, with a focus on the split between hard SQL filters and softer search signals.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.evaluate_query_parser import evaluate
from src.real_estate_nlp.query_parser import QueryParser
from src.real_estate_nlp.schema_validator import SchemaValidator

---

## 1. Artifacts Loading

Load the Week 1 query set and the Week 4 parser utilities. The query labels act as the main evaluation target.

In [2]:
with open("../data/processed/sample_queries.json") as f:
    queries = json.load(f)

parser = QueryParser()
validator = SchemaValidator()
query_df = pd.DataFrame(queries)

len(query_df)

120

---

## 2. Query Set Profile

Start with the labeled query set. The parser should handle both clean search filters and looser language that may become ranking signals later.

### 2.1 Intent and Difficulty

The sample covers simple filters, multi-constraint searches, and broader user goals.

In [3]:
pd.crosstab(query_df["intent"], query_df["difficulty"], margins=True)

difficulty,hard,medium,simple,All
intent,,,,
amenity_search,5,3,4,12
bed_bath_filter,4,3,3,10
condition_search,3,3,4,10
exterior_feature_search,3,3,4,10
interior_feature_search,3,3,4,10
investment_search,4,3,3,10
location_search,4,3,3,10
open_house_search,3,3,2,8
price_filter,4,3,3,10


### 2.2 Expected Entity Fields

This gives a quick view of what the parser is expected to recover from the labeled examples.

In [4]:
field_counts = pd.Series(
    field
    for entities in query_df["entities"]
    for field in entities
).value_counts()

field_counts.to_frame("count")

,count
city,22
property_type,19
amenities,16
condition,16
max_price,15
location_features,15
interior_features,11
investment_features,10
exterior_features,10
summary_focus,10


---

## 3. Parser Output Structure

The parser now separates executable database filters from softer search signals.

- `hard_filters`: safe to use in SQL WHERE clauses, including price, beds, baths, sqft, city, and confirmed database flags.
- `soft_signals`: useful for ranking, semantic search, or later result explanation.
- `filters`: the full flat parse, kept for inspection.

In [5]:
example_queries = [
    "3 bed homes in Irvine under 900k with pool",
    "homes in Newport Beach",
    "not interested in condos, show detached homes",
    "anything with canyon or ocean views but not too far from the freeway",
]

pd.DataFrame(
    {
        "query": query,
        "intent": parser.parse(query)["intent"],
        "hard_filters": parser.parse(query)["hard_filters"],
        "soft_signals": parser.parse(query)["soft_signals"],
    }
    for query in example_queries
)

,query,intent,hard_filters,soft_signals
0,3 bed homes in Irvine under 900k with pool,amenity_search,"{'price_max': 900000, 'beds_min': 3, 'city': '...",{'amenities': ['pool']}
1,homes in Newport Beach,location_search,{'city': 'Newport Beach'},{}
2,"not interested in condos, show detached homes",property_search,{},"{'property_type_exclude': ['condo'], 'property..."
3,anything with canyon or ocean views but not to...,location_search,{'has_view': True},"{'location_features': ['canyon view', 'ocean v..."


The split keeps broad recall intact. Some features, such as private pool, fireplace, and view, are promoted to hard filters only when a reliable database flag exists.

---

## 4. Evaluation

The default evaluation follows the search execution path: hard filters are treated as the exact-match target, while soft signals are inspected separately.

### 4.1 Default Hard-Filter Metrics

These metrics answer whether the parser captures the fields that would drive SQL retrieval.

In [6]:
report = evaluate(queries, parser)

pd.DataFrame(
    [
        {"metric": "hard_filter_exact_match_rate", "value": report["hard_filter_exact_match_rate"]},
        {"metric": "matched_expected_fields", "value": report["matched_expected_fields"]},
        {"metric": "total_expected_fields", "value": report["total_expected_fields"]},
    ]
)


,metric,value
0,hard_filter_exact_match_rate,1.0
1,matched_expected_fields,205.0
2,total_expected_fields,205.0


At this stage, the hard-filter path is clean. The parser covers the labeled SQL-style fields without adding extra hard constraints.

### 4.2 Soft-Signal Diagnostics

Soft signals are allowed to be richer than the gold labels, but the extra fields are still worth reviewing.

In [7]:
soft_report = evaluate(queries, parser, include_soft_signals=True)

pd.DataFrame(
    [
        {"metric": "hard_filter_exact_match_rate", "value": soft_report["hard_filter_exact_match_rate"]},
        {"metric": "soft_signal_exact_match_rate", "value": soft_report["soft_signal_exact_match_rate"]},
        {"metric": "full_filter_exact_match_rate", "value": soft_report["full_filter_exact_match_rate"]},
    ]
)


,metric,value
0,hard_filter_exact_match_rate,1.000000
1,soft_signal_exact_match_rate,1.000000
2,full_filter_exact_match_rate,0.916667


---

## 5. SQL and Validation

The SQL generator uses hard filters by default. Soft signals can be included only when explicitly requested.

### 5.1 Parameterized SQL

The query text does not get concatenated into SQL. User-derived values stay in the params list.

In [8]:
parsed = parser.parse("3 bed homes in Irvine under 900k with pool")
sql, params = parser.to_sql(parsed)

sql, params

('SELECT * FROM rets_property WHERE L_City = %s AND L_SystemPrice <= %s AND L_Keyword2 >= %s AND PoolPrivateYN = %s',
 ['Irvine', 900000, 3, True])

In [9]:
sql_with_soft, params_with_soft = parser.to_sql(parsed, include_soft_signals=True)

sql_with_soft, params_with_soft

("SELECT * FROM rets_property WHERE L_City = %s AND L_SystemPrice <= %s AND L_Keyword2 >= %s AND PoolPrivateYN = %s AND L_Remarks LIKE %s ESCAPE '\\\\'",
 ['Irvine', 900000, 3, True, '%pool%'])

### 5.2 Injection Check

The parser should not pass malicious text into the SQL template.

In [10]:
attack = "homes in Irvine'; DROP TABLE rets_property; -- under 900k"
attack_sql, attack_params = parser.to_sql(parser.parse(attack))

{"DROP TABLE in sql": "DROP TABLE" in attack_sql, "sql": attack_sql, "params": attack_params}

{'DROP TABLE in sql': False,
 'sql': 'SELECT * FROM rets_property WHERE L_City = %s AND L_SystemPrice <= %s',
 'params': ['Irvine', 900000]}

### 5.3 Validation Examples

Validation catches values that parse cleanly but should not be sent to the database.

In [11]:
validation_examples = [
    {"case": "valid", "filters": parser.parse("homes in Irvine under 900k")},
    {"case": "unknown city", "filters": {"city": "Atlantis"}},
    {"case": "bad price", "filters": {"price_max": 49_999}},
    {"case": "bad bed count", "filters": {"beds_min": 99}},
]

pd.DataFrame(
    {
        "case": item["case"],
        "valid": validator.validate_query(item["filters"])[0],
        "errors": validator.validate_query(item["filters"])[1],
    }
    for item in validation_examples
)

,case,valid,errors
0,valid,True,[]
1,unknown city,False,[City 'Atlantis' not found in known city list]
2,bad price,False,[price_max=49999 is outside the supported range]
3,bad bed count,False,[beds_min=99 is outside the supported range]
